# Chương 3 — Ứng dụng 2: Bag of Visual Words

**Dữ liệu thực:** UCI Optical Recognition of Handwritten Digits (bản 1.797 ảnh 8×8 đi kèm `scikit-learn`; sử dụng 4 lớp 0, 1, 3, 8). **Thuật toán FIM:** negFIN Julia của nhóm (không dùng FIM library).

Quy trình: ảnh → phóng kích thước → trích dense SIFT descriptors → học codebook bằng K-means trên tập train → chuyển ảnh thành giao dịch gồm ID visual words → chạy negFIN trên train → tìm pattern nổi bật theo từng nhãn → đánh giá thêm pattern làm đặc trưng cho bộ phân loại đơn giản trên test chưa từng thấy.

Nguồn: [scikit-learn load_digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html). SIFT và K-means chỉ chuẩn bị biểu diễn ảnh, **không thay thế negFIN**.

**Lưu ý Colab:** sau khi push source, gán `REPO_URL` ở ô tiếp theo; nếu chạy trong thư mục project, giữ rỗng.

In [ ]:
# Tự dùng code trong repository hoặc clone repo chính thức (sau khi đã push Chương 3).
from pathlib import Path
import os, subprocess, sys, shutil, platform

REPO_URL = "https://github.com/Thanhtung46/Frequent_Itemset_Mining.git"
PROJECT_DIR = None
for directory in [Path.cwd(), Path.cwd() / 'Frequent_Itemset_Mining', Path('/content/Frequent_Itemset_Mining')]:
    if (directory / 'src' / 'algorithm' / 'negfin_baseline.jl').exists() and (directory / 'applications').is_dir():
        PROJECT_DIR = directory.resolve()
        break
if PROJECT_DIR is None:
    if not REPO_URL.startswith('https://github.com/'):
        raise ValueError('Đặt REPO_URL tới repo GitHub public của nhóm.')
    PROJECT_DIR = Path('/content/Frequent_Itemset_Mining')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
if not (PROJECT_DIR / 'applications' / 'common.py').exists():
    raise RuntimeError('Repo GitHub chưa được push phần Chương 3. Hãy commit/push mã nguồn mới.')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Project directory:', PROJECT_DIR)


In [ ]:
# Cài gói Python hỗ trợ đọc dữ liệu/biểu đồ/ảnh; KHÔNG cài gói khai thác FIM.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-ch3.txt'], check=True)

# Cài Julia nếu đang dùng Colab Linux. Máy Windows/Mac: cài Julia >= 1.9 trước.
if shutil.which(os.environ.get('JULIA_BIN', 'julia')) is None:
    if sys.platform == 'linux' and platform.machine() in ('x86_64', 'AMD64'):
        julia_home = Path('/content/julia-1.11.4') if Path('/content').is_dir() else Path.home() / '.cache' / 'negfin-julia-1.11.4'
        julia_exe = julia_home / 'bin' / 'julia'
        if not julia_exe.exists():
            command = ('curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.11/'
                       'julia-1.11.4-linux-x86_64.tar.gz -o /tmp/negfin-julia.tar.gz'
                       ' && mkdir -p ' + str(julia_home)
                       + ' && tar -xzf /tmp/negfin-julia.tar.gz -C ' + str(julia_home)
                       + ' --strip-components=1')
            subprocess.run(['bash', '-lc', command], check=True)
        os.environ['JULIA_BIN'] = str(julia_exe)
    else:
        raise RuntimeError('Cần cài Julia >= 1.9 và đưa julia vào PATH hoặc đặt biến JULIA_BIN.')
subprocess.run([os.environ.get('JULIA_BIN', 'julia'), '--version'], check=True)
print('Dùng negFIN của nhóm:', PROJECT_DIR / 'src' / 'algorithm' / 'negfin_baseline.jl')

## 1. Quyết định biến đổi ảnh thành transaction

Vì ảnh gốc chỉ 8×8 pixel, ta phóng thành 96×96 và trích **dense SIFT** tại lưới 4×4 cố định để giảm trường hợp thiếu keypoint. Mỗi descriptor 128 chiều được ánh xạ tới visual word gần nhất trong codebook 24 tâm cụm K-means. Một ảnh → một giao dịch chứa **tập hợp visual words xuất hiện**; không dùng nhãn lớp làm item để tránh rò rỉ nhãn.

Chia **train/test 75/25** có stratification, `random_state=42` trước khi học codebook. Chỉ dùng train để fit K-means, khai thác itemset, chọn pattern và fit classifier. Chỉ dùng test để đo độ chính xác.

In [ ]:
from applications.visual_words import prepare_images, descriptors_for_image, SEED
from IPython.display import display
import matplotlib.pyplot as plt
train_img, test_img, train_y, test_y = prepare_images()
print('Train:', train_img.shape, 'Test:', test_img.shape, 'Classes:', sorted(set(train_y)))
print('SIFT dense của ảnh đầu:', descriptors_for_image(train_img[0]).shape)
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, img, label in zip(axes, train_img[:6], train_y[:6]):
    ax.imshow(img, cmap='gray_r', interpolation='nearest')
    ax.set_title(f'Label {label}')
    ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Chạy toàn bộ pipeline và đối chiếu support

Chỉ dùng thư viện `cv2`/`sklearn` cho image descriptor, K-means và classifier. Tập frequent itemsets được **chính Julia `negfin_baseline`** khai thác. Chương trình đối chiếu `support_count` của mỗi itemset trả về với số ảnh chứa pattern trong train trước khi phân tích tiếp.

In [ ]:
from applications.run_visual import execute
MIN_SUPPORT_VISUAL = 0.08
CODEBOOK_SIZE = 24
visual_result = execute(minsup=MIN_SUPPORT_VISUAL, n_words=CODEBOOK_SIZE)
display(visual_result['enrichment'].head(12))
display(visual_result['evaluation'])

## 3. Diễn giải pattern trực quan

`W1`, `W2`, ... là chỉ số tâm cụm SIFT, **không tự động tương ứng với các nét chữ cố định**. Mức đặc trưng (*class enrichment*) tính bằng `P(pattern | class) / P(pattern)`, không phải điểm chính xác phân lớp. Quan sát mức enrichment cao và số mẫu hỗ trợ của pattern; tránh khẳng định đó là đặc điểm mang tính nhân quả hoặc có tính phổ quát trên tập ảnh khác.

In [ ]:
from IPython.display import Image
folder = visual_result['work_dir']
display(Image(filename=str(folder / 'sample_digits.png')))
if (folder / 'enriched_patterns.png').exists():
    display(Image(filename=str(folder / 'enriched_patterns.png')))
print('Các pattern đặc trưng theo nhãn:')
print(visual_result['enrichment'].head(10).to_string(index=False))

## 4. Đánh giá mô hình downstream trên tập test

Dùng cùng Logistic Regression và cùng dữ liệu chia train/test, chỉ khác bộ đặc trưng (binary visual words so với binary visual words kết hợp frequent patterns). Nếu độ chính xác không cải thiện, vẫn phải báo cáo trung thực: các pattern phổ biến có thể dư thừa, SIFT trên ảnh 8×8 được phóng to có giới hạn và tập Digits không tương đồng ảnh tự nhiên.

Kết quả CSV/PNG tại `results/chapter3/visual_words/`. Mẫu đồ án minh họa một ứng dụng dễ tái lập trên dữ liệu thật; không khẳng định tương đương thí nghiệm CIFAR-10 hoặc Caltech-101.